In [1]:
# Imports and Login
import os, torch, json, sqlite3
import pandas as pd
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from datasets import load_dataset

HF_TOKEN = "Your Key Here" 
login(token=HF_TOKEN)

BASE_MODEL  = "meta-llama/Llama-3.2-3B-Instruct"
LORA_MODEL  = "Krupa1420/text2sql-llama3-qlora"
DATASET     = "Krupa1420/text2sqldata"

print("Ready!")

/opt/conda/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.0' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/conda/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.4.0' currently installed).
  from pandas.core import (


Ready!


In [4]:
import torch
from unsloth import FastLanguageModel
import warnings
warnings.filterwarnings("ignore")

ft_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "./text2sql-run-1/checkpoint-1100",
    max_seq_length = 2048,
    dtype          = torch.float16,
    load_in_4bit   = True,
)

FastLanguageModel.for_inference(ft_model)
print("Model loaded!")

BIKESHARE_SCHEMA = """CREATE TABLE trips (
  trip_id INTEGER, start_time TIMESTAMP, end_time TIMESTAMP,
  duration_minutes FLOAT, start_station_id VARCHAR,
  end_station_id VARCHAR, bike_id VARCHAR,
  bike_type VARCHAR, passholder_type VARCHAR
);
CREATE TABLE stations (
  station_id VARCHAR, station_name VARCHAR, lat FLOAT, lon FLOAT
);
CREATE TABLE weather (
  timestamp TIMESTAMP, temp_c FLOAT, precip_mm FLOAT,
  wind_speed_mps FLOAT, rel_humidity FLOAT,
  pressure_hpa FLOAT, weather_code INTEGER
);"""

def predict_sql(schema, question, max_new_tokens=128):
    prompt = f"""### Instruction:
You are an expert SQL assistant. Given a database schema and a natural language question, write a correct SQL query. Only output the SQL query, no explanation.

### Schema:
{schema}

### Question:
{question}

### SQL:
"""
    inputs = tokenizer(
        [prompt],
        return_tensors     = "pt",
        truncation         = True,
        max_length         = 512,
        add_special_tokens = True,
    ).to("cuda")

    with torch.no_grad():
        outputs = ft_model.generate(
            input_ids      = inputs['input_ids'],
            attention_mask = inputs['attention_mask'],
            max_new_tokens = max_new_tokens,
            do_sample      = False,
            use_cache      = False,    # ← disable KV cache
            pad_token_id   = tokenizer.eos_token_id,
        )

    prompt_len = inputs['input_ids'].shape[1]
    generated  = outputs[0][prompt_len:]
    sql = tokenizer.decode(generated, skip_special_tokens=True)
    sql = sql.split("###")[0].strip()
    return sql

# Test
questions = [
    "How many electric bike trips were taken in July?",
    "Which station had the most trips?",
    "What is the average trip duration on weekends?",
    "How many trips were taken on rainy days?",
]

print("=" * 60)
for q in questions:
    sql = predict_sql(BIKESHARE_SCHEMA, q)
    print(f"Q  : {q}")
    print(f"SQL: {sql}")
    print("-" * 60)

==((====))==  Unsloth 2026.3.8: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    NVIDIA L40. Num GPUs = 2. Max memory: 44.394 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.3.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.
Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Model loaded!


Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q  : How many electric bike trips were taken in July?
SQL: SELECT COUNT(*) FROM trips WHERE bike_type = 'Electric' AND MONTH(start_time) = 7;
------------------------------------------------------------


Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q  : Which station had the most trips?
SQL: SELECT s.station_name, COUNT(*) as trips
FROM trips t JOIN stations s ON t.start_station_id = s.station_id
GROUP BY s.station_name ORDER BY trips DESC LIMIT 1;
------------------------------------------------------------


Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q  : What is the average trip duration on weekends?
SQL: SELECT AVG(duration_minutes) FROM trips WHERE DAYOFWEEK(start_time) IN (1, 7);
------------------------------------------------------------
Q  : How many trips were taken on rainy days?
SQL: SELECT COUNT(*) FROM trips t JOIN weather w ON DATE(t.start_time) = DATE(w.timestamp) WHERE w.precip_mm > 0;
------------------------------------------------------------


In [5]:
# Push best checkpoint to HuggingFace
HUB_OUT  = "Krupa1420/text2sql-llama3-qlora"

ft_model.push_to_hub(HUB_OUT, token=HF_TOKEN)
tokenizer.push_to_hub(HUB_OUT, token=HF_TOKEN)
print(f"Best model live at: https://huggingface.co/{HUB_OUT}")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/Krupa1420/text2sql-llama3-qlora


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Best model live at: https://huggingface.co/Krupa1420/text2sql-llama3-qlora


In [6]:
import torch, sqlite3
from unsloth import FastLanguageModel

# Load model
ft_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "./text2sql-run-1/checkpoint-1100",
    max_seq_length = 2048,
    dtype          = torch.float16,
    load_in_4bit   = True,
)
FastLanguageModel.for_inference(ft_model)
print("Model loaded!")

BIKESHARE_SCHEMA = """CREATE TABLE trips (
  trip_id INTEGER, start_time TIMESTAMP, end_time TIMESTAMP,
  duration_minutes FLOAT, start_station_id VARCHAR,
  end_station_id VARCHAR, bike_id VARCHAR,
  bike_type VARCHAR, passholder_type VARCHAR
);
CREATE TABLE stations (
  station_id VARCHAR, station_name VARCHAR, lat FLOAT, lon FLOAT
);
CREATE TABLE weather (
  timestamp TIMESTAMP, temp_c FLOAT, precip_mm FLOAT,
  wind_speed_mps FLOAT, rel_humidity FLOAT,
  pressure_hpa FLOAT, weather_code INTEGER
);"""

def predict_sql(schema, question, max_new_tokens=128):
    prompt = f"""### Instruction:
You are an expert SQL assistant. Given a database schema and a natural language question, write a correct SQL query. Only output the SQL query, no explanation.

### Schema:
{schema}

### Question:
{question}

### SQL:
"""
    inputs = tokenizer(
        [prompt],
        return_tensors     = "pt",
        truncation         = True,
        max_length         = 512,
        add_special_tokens = True,
    ).to("cuda")

    with torch.no_grad():
        outputs = ft_model.generate(
            input_ids      = inputs['input_ids'],
            attention_mask = inputs['attention_mask'],
            max_new_tokens = max_new_tokens,
            do_sample      = False,
            use_cache      = False,
            pad_token_id   = tokenizer.eos_token_id,
        )

    prompt_len = inputs['input_ids'].shape[1]
    generated  = outputs[0][prompt_len:]
    sql = tokenizer.decode(generated, skip_special_tokens=True)
    sql = sql.split("###")[0].strip()
    return sql

# Quick test — 10 bikeshare questions
print("\n" + "="*60)
print("QUICK MODEL TEST — BIKESHARE QUESTIONS")
print("="*60)

test_questions = [
    "How many trips were taken in total?",
    "How many electric bike trips were taken in July?",
    "What is the average trip duration on weekends?",
    "Which station had the most trips?",
    "How many trips were taken on rainy days?",
    "What are the top 5 busiest stations?",
    "Which month has the most trips?",
    "What is the busiest hour of the day?",
    "Which passholder type takes the longest trips?",
    "What is the running total of trips by month?",
]

for i, q in enumerate(test_questions):
    sql = predict_sql(BIKESHARE_SCHEMA, q)
    print(f"\n[{i+1}] Q  : {q}")
    print(f"     SQL: {sql}")

print("\nQuick test done!")
print("If SQLs look correct to run full evaluation!")

==((====))==  Unsloth 2026.3.8: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    NVIDIA L40. Num GPUs = 2. Max memory: 44.394 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ Model loaded!

QUICK MODEL TEST — BIKESHARE QUESTIONS


Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


[7] Q  : Which month has the most trips?
     SQL: SELECT MONTH(start_time) as month, COUNT(*) as trip_count
FROM trips GROUP BY month ORDER BY trip_count DESC LIMIT 1;


Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[8] Q  : What is the busiest hour of the day?
     SQL: SELECT HOUR(start_time) as hour, COUNT(*) as trips
FROM trips GROUP BY hour ORDER BY trips DESC LIMIT 1; -- get the hour with the highest count; in case of ties, return the first one. -- Note: This assumes that the system is up and running for at least a day. If not, you might need to adjust the query.


Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[9] Q  : Which passholder type takes the longest trips?
     SQL: SELECT passholder_type, AVG(duration_minutes) as avg_duration
FROM trips GROUP BY passholder_type ORDER BY avg_duration DESC LIMIT 1;

[10] Q  : What is the running total of trips by month?
     SQL: SELECT month, monthly_trips,
  SUM(monthly_trips) OVER (ORDER BY month) as running_total
FROM (
  SELECT MONTH(start_time) as month, COUNT(*) as monthly_trips
  FROM trips GROUP BY MONTH(start_time)
) monthly
ORDER BY month;

✅ Quick test done!
If SQLs look correct → run full evaluation!


In [10]:
import json, sqlite3
from datasets import load_dataset

# Load OUR dataset from HuggingFace
ds  = load_dataset("Krupa1420/text2sqldata")
val = ds['validation']

print(f" Validation set loaded!")
print(f"   Total     : {len(val):,}")
print(f"   Spider    : {val['source'].count('spider'):,}")
print(f"   Bikeshare : {val['source'].count('bikeshare'):,}")

 Validation set loaded!
   Total     : 730
   Spider    : 694
   Bikeshare : 36


In [12]:
# Exact match function
def exact_match(pred, gold):
    pred = pred.strip().rstrip(';').upper().replace('  ', ' ').strip()
    gold = gold.strip().rstrip(';').upper().replace('  ', ' ').strip()
    return pred == gold

# Partial match function
def partial_match(pred, gold):
    pred_u = pred.upper()
    gold_u = gold.upper()
    score, checks = 0, 0
    for kw in ['SELECT', 'FROM', 'WHERE', 'JOIN',
               'GROUP BY', 'ORDER BY', 'HAVING', 'LIMIT']:
        if kw in gold_u:
            checks += 1
            if kw in pred_u:
                score += 1
    return score / checks if checks > 0 else 0

# Run evaluation
print(" Evaluating on 730 validation examples...")

results = {
    'spider'   : {'exact': 0, 'total': 0, 'partial': []},
    'bikeshare': {'exact': 0, 'total': 0, 'partial': []}
}

for i, ex in enumerate(val):
    # Extract schema from text
    text     = ex['text']
    schema_s = text.find('### Schema:') + len('### Schema:')
    schema_e = text.find('### Question:')
    schema   = text[schema_s:schema_e].strip()

    pred   = predict_sql(schema, ex['question'])
    gold   = ex['sql']
    source = ex['source']

    em = exact_match(pred, gold)
    pm = partial_match(pred, gold)

    results[source]['total']   += 1
    results[source]['exact']   += int(em)
    results[source]['partial'].append(pm)

    if (i+1) % 100 == 0:
        s_em = results['spider']['exact'] / max(results['spider']['total'], 1) * 100
        b_em = results['bikeshare']['exact'] / max(results['bikeshare']['total'], 1) * 100
        print(f"  [{i+1}/730] Spider EM: {s_em:.1f}% | Bikeshare EM: {b_em:.1f}%")

# Final results
print("FINAL EVALUATION RESULTS")


for source in ['spider', 'bikeshare']:
    r     = results[source]
    total = r['total']
    exact = r['exact']
    part  = sum(r['partial']) / len(r['partial']) * 100
    print(f"\n{source.upper()} ({total} examples):")
    print(f"   Exact Match   : {exact}/{total} = {exact/total*100:.1f}%")
    print(f"   Partial Match : {part:.1f}%")

total_ex  = results['spider']['exact'] + results['bikeshare']['exact']
total_all = results['spider']['total']  + results['bikeshare']['total']
all_part  = (sum(results['spider']['partial']) +
             sum(results['bikeshare']['partial'])) / total_all * 100

print(f"\nOVERALL ({total_all} examples):")
print(f"   Exact Match   : {total_ex}/{total_all} = {total_ex/total_all*100:.1f}%")
print(f"   Partial Match : {all_part:.1f}%")


# Show 3 correct and 3 wrong examples
print("\n 3 CORRECT PREDICTIONS:")
correct_count = 0
for ex in val:
    if correct_count >= 3:
        break
    text     = ex['text']
    schema_s = text.find('### Schema:') + len('### Schema:')
    schema_e = text.find('### Question:')
    schema   = text[schema_s:schema_e].strip()
    pred     = predict_sql(schema, ex['question'])
    if exact_match(pred, ex['sql']):
        print(f"\n   Q       : {ex['question']}")
        print(f"   Gold    : {ex['sql']}")
        print(f"   Pred    : {pred}")
        correct_count += 1

print("\n 3 WRONG PREDICTIONS:")
wrong_count = 0
for ex in val:
    if wrong_count >= 3:
        break
    text     = ex['text']
    schema_s = text.find('### Schema:') + len('### Schema:')
    schema_e = text.find('### Question:')
    schema   = text[schema_s:schema_e].strip()
    pred     = predict_sql(schema, ex['question'])
    if not exact_match(pred, ex['sql']):
        print(f"\n   Q       : {ex['question']}")
        print(f"   Gold    : {ex['sql']}")
        print(f"   Pred    : {pred}")
        wrong_count += 1
